In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import copernicusmarine as cm

sys.path.insert(0, ".")
from src.barrier_layers import compute_segment_gradient, first_depth_below_threshold

In [4]:
ds = cm.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",

    dataset_version="202311",
    variables=["mlotst", "thetao", "so"],
    minimum_longitude=-180, maximum_longitude=179.9166717529297,
    minimum_latitude=-80, maximum_latitude=90,
    start_datetime="2023-12-30T00:00:00",
    end_datetime="2023-12-30T23:59:59",
    minimum_depth=0.49402499198913574,
    maximum_depth=541.0889282226562,
    coordinates_selection_method="strict-inside",
    # username="dbrisaro@gmail.com",
    # password="Clarita1905.!!",
)

INFO - 2026-01-05T20:39:23Z - Selected dataset version: "202311"
INFO - 2026-01-05T20:39:23Z - Selected dataset part: "default"


In [14]:
latitude = ds.latitude.values
longitude = ds.longitude.values

In [17]:
from scipy.io import savemat

# Save longitude and latitude values to .mat file
data_to_save = {
    'longitude': longitude[0::12],
    'latitude': latitude[0::12]
}

savemat('coordinates.mat', data_to_save)
print("Longitude and latitude values saved to 'coordinates.mat'")

Longitude and latitude values saved to 'coordinates.mat'


In [18]:
data_to_save

{'longitude': array([-180., -179., -178., -177., -176., -175., -174., -173., -172.,
        -171., -170., -169., -168., -167., -166., -165., -164., -163.,
        -162., -161., -160., -159., -158., -157., -156., -155., -154.,
        -153., -152., -151., -150., -149., -148., -147., -146., -145.,
        -144., -143., -142., -141., -140., -139., -138., -137., -136.,
        -135., -134., -133., -132., -131., -130., -129., -128., -127.,
        -126., -125., -124., -123., -122., -121., -120., -119., -118.,
        -117., -116., -115., -114., -113., -112., -111., -110., -109.,
        -108., -107., -106., -105., -104., -103., -102., -101., -100.,
         -99.,  -98.,  -97.,  -96.,  -95.,  -94.,  -93.,  -92.,  -91.,
         -90.,  -89.,  -88.,  -87.,  -86.,  -85.,  -84.,  -83.,  -82.,
         -81.,  -80.,  -79.,  -78.,  -77.,  -76.,  -75.,  -74.,  -73.,
         -72.,  -71.,  -70.,  -69.,  -68.,  -67.,  -66.,  -65.,  -64.,
         -63.,  -62.,  -61.,  -60.,  -59.,  -58.,  -57.,  -56., 

In [ ]:
point_longitude = 55.0
point_latitude = -10.0
threshold = -0.1

ds_point = ds.isel(time=0).sel(latitude=point_latitude).sel(longitude=point_longitude)
lat = ds_point["latitude"].values
lon = ds_point["longitude"].values

mld = ds_point["mlotst"].values
salinity_profile = ds_point["so"].values
temperature_profile = ds_point["thetao"].values
depth = ds_point["depth"].values

depth_mid, gradient = compute_segment_gradient(temperature_profile, depth)
ild, grad_at_ild = first_depth_below_threshold(gradient, depth_mid, threshold=threshold)

if ild is not None:
    bld = ild - mld

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = plt.axes([0.05, 0.05, 0.4, 0.9])
bx = ax.twiny()

ax.plot(temperature_profile, -depth, ".", color="salmon", markersize=3, linewidth=0.5, linestyle="-")
bx.plot(salinity_profile, -depth, ".", color="black", markersize=3, linewidth=0.5, linestyle="-")

if ild is not None:
    ax.axhline(y=-ild, color="lightblue", linewidth=1.5, linestyle="--")
    ax.text(temperature_profile.min(), -ild + 10, f"ILD: {ild:.1f}m", va="center", ha="left", color="lightblue", fontsize=9)
    ax.set_title(
        f"Vertical profile (0.083°) — Lat: {lat:.1f}°N, Lon: {lon:.1f}°E\n"
        f"Threshold: {threshold:.1f}°C/m  |  MLD: {mld:.1f}m  ILD: {ild:.1f}m  BLD: {bld:.1f}m",
        fontsize=11, pad=20,
    )
else:
    ax.set_title(
        f"Vertical profile (0.083°) — Lat: {lat:.1f}°N, Lon: {lon:.1f}°E\n"
        f"Threshold: {threshold:.1f}°C/m  |  MLD: {mld:.1f}m  ILD: undefined",
        fontsize=11, pad=20,
    )

ax.axhline(y=-mld, color="orange", linewidth=1.5, linestyle="--")
ax.text(temperature_profile.min(), -mld + 10, f"MLD: {mld:.1f}m", va="center", ha="left", color="orange", fontsize=9)

ax.set_xlabel("Temperature (°C)", color="salmon")
bx.set_xlabel("Salinity (psu)", color="black")
ax.set_ylabel("Depth (m)")
ax.set_ylim([-500, 0])
bx.set_xlim([34.65, 35.4])
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.spines["bottom"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", colors="salmon")
ax.spines["top"].set_color("salmon")
bx.xaxis.set_ticks_position("top")
bx.xaxis.set_label_position("top")
bx.spines["bottom"].set_visible(False)
bx.spines["right"].set_visible(False)
bx.spines["top"].set_position(("outward", 40))
bx.spines["top"].set_color("black")
bx.tick_params(axis="x", colors="black")
bx.xaxis.label.set_color("black")

cx = plt.axes([0.5, 0.05, 0.4, 0.9])
cx.plot(gradient, -depth_mid, ".", color="darkgrey", markersize=3, linewidth=0.5, linestyle="-")
cx.set_title("Temperature gradient", fontsize=11, pad=20)
cx.set_xlabel("dT/dz (°C/m)")
cx.set_ylim([-500, 0])
cx.xaxis.set_ticks_position("top")
cx.xaxis.set_label_position("top")
cx.spines["bottom"].set_visible(False)
cx.spines["right"].set_visible(False)
cx.axvline(x=threshold, color="black", linewidth=0.5, linestyle="-.")
cx.set_xlim([-0.3, 0])

date_str = ds.time.values[0].astype("datetime64[D]").astype(str)
fig.savefig(
    f"displays/profile_083deg_{date_str}_lat_{lat:.1f}_lon_{lon:.1f}_threshold_{threshold:.1f}.png",
    bbox_inches="tight", dpi=400,
)